In [1]:
import pandas as pd
import json
import time
import re
from datetime import datetime
from sklearn.metrics import classification_report, accuracy_score
from tenacity import retry, stop_after_attempt, wait_exponential
from openai import OpenAI

# ==========================================
# 1. Data clearning
# ==========================================

def load_and_clean_data(file_path):
    df = pd.read_csv(file_path)
    
    def to_float(val):
        if pd.isna(val): return None
        if isinstance(val, str):
            has_pct = '%' in val
            val_clean = val.replace('%', '').replace('+', '').strip()
            try:
                return float(val_clean) / 100.0 if has_pct else float(val_clean)
            except ValueError:
                return None
        return val

    df['rev_val'] = df['total_yoy_growth'].apply(to_float)
    df['stock_val'] = df['stock_growth'].apply(to_float)
    
    df['true_stock_label'] = df['stock_val'].apply(lambda x: "Growth" if x > 0 else "Loss")
    df['true_segment_label'] = df['max_contribution_segment_growth'].apply(lambda x: "Growth" if (x is not None and x > 0) else "Loss")
    
    return df.dropna(subset=['rev_val', 'stock_val'])


# ==========================================
# 2. Prompt Design
# ==========================================

SYSTEM_INSTRUCTION = "You are a professional financial analyst. You must analyze the data and return a prediction in JSON format. Output MUST be a valid JSON object."

PROMPT_NORMAL = """
Based on a year's worth of corporate tweets, predict the annual stock price movement for {company} ({year}).

[Tweet Corpus]
{tweets}

Output Requirement: Return ONLY a JSON object: {{"stock_prediction": "Growth" or "Loss"}}
"""

PROMPT_COT = """
Predict stock movement for {company} ({year}) using this logical chain:
1. Analyze signals for the core revenue segment: "{max_segment}".
2. Predict segment performance and determine if it aligns with the overall stock trend.

[Tweet Corpus]
{tweets}

Output Requirement: Return ONLY a JSON object: 
{{"segment_prediction": "Growth" or "Loss", "stock_prediction": "Growth" or "Loss"}}
"""

PROMPT_QCOT = """
Identify if "{company}" stock will result in "Growth" or "Loss" for {year}.
Industry: {industry_type} | Segments: {segments}
Tweets: {tweets}

[Decision Logic - CRITICAL]
1. Weights: Estimate revenue % per segment (total 1.0).
2. Scores: Rate segment tweets from -1.0 to 1.0. 
   - Rule A: PR hype/Marketing/Events without concrete growth metrics (e.g., revenue %, order volume) MUST be scored 0.0 or -0.1.
   - Rule B: Any mention of "macro headwinds", "restructuring", or "challenging environment" must be scored < -0.5.
3. Threshold: If the final 'weighted_sum' is below {threshold}, the outlook is stagnant/negative; you MUST predict "Loss".

- Be concise. Focus ONLY on High-Impact market signals.
- Do not provide conversational filler.

Return EXACTLY this JSON format:
{{
  "stock_prediction": "Growth" or "Loss",
  "weights": "segment1: weight, segment2: weight...",
  "scores": "segment1: score, segment2: score...",
  "weighted_sum": float,
  "logic_brief": "One sentence summary of the primary driver",
  "threshold_met": "Is weighted_sum > {threshold}? (Yes/No)"
}}
"""

# ==========================================
# 3. 实验执行逻辑
# ==========================================

@retry(
    stop=stop_after_attempt(2), 
    wait=wait_exponential(multiplier=1, min=4, max=60),
    before_sleep=lambda retry_state: print(f"Error, trying {retry_state.attempt_number} ..."),
    reraise=True
)
def get_model_prediction_with_retry(prompt, model):
    response = model.generate_content(prompt)
    
    text = response.text
    
    clean_text = re.sub(r'```json\s?|```', '', text).strip()
    clean_text = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', clean_text)
    
    try:
        result = json.loads(clean_text, strict=False)
    except json.JSONDecodeError:
        # If the JSON parsing fails, try to extract the content within the first curly brace using regular expressions.
        match = re.search(r'\{.*\}', clean_text, re.DOTALL)
        if match:
            result = json.loads(match.group(0))
        else:
            raise ValueError(f"Failed to parse JSON from response: {clean_text[:100]}")
    
    stock_pred = result.get("stock_prediction", "Error")
    seg_pred = result.get("segment_prediction", "N/A")
    final_score = result.get("weighted_sum", 0)
    reasoning = result.get("logic_brief", "Error")
    
    return {"stock_prediction": stock_pred, "segment_prediction": seg_pred, "final_score": final_score, "reasoning": reasoning}


def get_model_prediction(prompt, model):
    try:
        return get_model_prediction_with_retry(prompt, model)
    except Exception as e:
        print(f"API Error: {e}")
        return {"stock_prediction": "Error", "segment_prediction": "Error", "final_score": 0, "reasoning": "Error"}


def run_experiment(twitter_df, df, model, threshold=0.25, mode="cot", max_num=1000):
    if not pd.api.types.is_datetime64_any_dtype(twitter_df['release_time']):
        twitter_df['release_time'] = pd.to_datetime(twitter_df['release_time'])

    y_true_stock = []
    y_pred_stock = []
    
    print(f"Starting experiment in {mode.upper()} mode...")
    
    for idx, row in df.iterrows():
        company_name = row['company_name']
        if company_name == "Huawei":
            continue

        report_period = row['report_period']

        print(f"company_name: {company_name}, report_period: {report_period}")
        
        start_str, end_str = report_period.split(" to ")
        start_date = datetime.strptime(start_str.strip(), "%Y-%m-%d")
        end_date = datetime.strptime(end_str.strip(), "%Y-%m-%d")
        
        # Filtering tweets
        mask = (
            (twitter_df['name'] == company_name) & 
            (twitter_df['release_time'] >= start_date) & 
            (twitter_df['release_time'] <= end_date)
        )
        twitter_df['interactive_metric'] = twitter_df['num_comment'] + twitter_df['num_like'] + twitter_df['num_retweet']
        df_filtered = twitter_df[mask].copy()

        df_filtered = df_filtered[df_filtered['tweet'].str.len() <= 150]

        if len(df_filtered) > max_num:
            tweets_list = df_filtered['tweet'].sample(n=max_num, random_state=1).tolist()
        else:
            tweets_list = df_filtered['tweet'].tolist()
            
        formatted_tweets = "\n".join([f"- {t}" for t in tweets_list])
        
        if mode == "qcot":
            prompt = PROMPT_QCOT.format(
                company=company_name,
                year=row['fiscal_year'],
                industry_type=row['industry_type'],
                segments=row['segment_list'],
                tweets=formatted_tweets,
                threshold=threshold
            )
        elif mode == "cot":
            prompt = PROMPT_COT.format(
                company=company_name,
                year=row['fiscal_year'],
                max_segment=row['max_contribution_segment'],
                tweets=formatted_tweets
            )
        else:
            prompt = PROMPT_NORMAL.format(
                company=company_name,
                year=row['fiscal_year'],
                tweets=formatted_tweets
            )
            
        prediction = get_model_prediction(prompt, model)
        
        y_true_stock.append(row['true_stock_label'])
        y_pred_stock.append(prediction.get('stock_prediction', 'Error'))

        print(f"Company: {company_name} | Year: {row['fiscal_year']} | True: {row['true_stock_label']} | Pred: {prediction['stock_prediction']} | Score: {prediction['final_score']}")
        print(f"Reasoning: {prediction['reasoning']}")
        
        time.sleep(1)
        
    return y_true_stock, y_pred_stock

In [ ]:
import os
import google.generativeai as genai
import google.api_core.client_options
from google.api_core import retry

is_proxy = True  # If using VPN, set Network proxy.

if is_proxy: 
    proxy_url = 'http://127.0.0.1:7897'
    os.environ['HTTP_PROXY'] = proxy_url
    os.environ['HTTPS_PROXY'] = proxy_url
    os.environ['GRPC_PROXY_EXPAND_HOSTS'] = '1'
    os.environ['grpc_proxy'] = proxy_url


client_options = google.api_core.client_options.ClientOptions(
    api_endpoint="generativelanguage.googleapis.com"
)
genai.configure(
    api_key="API_KEY",  # your API Key
    transport='rest',
    client_options=client_options 
)

d:\anaconda3\envs\torch_cpu\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\PC\AppData\Local\Temp\ipykernel_40452\1790666984.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [4]:
# Dataset Statistics
csv_path = r"data/stock_cot.csv"
clean_df = load_and_clean_data(csv_path)
FILE_NAME_TWITTER = 'data/b2b_twitter.xlsx'
twitter_df = pd.read_excel(FILE_NAME_TWITTER)

correlation = clean_df['rev_val'].corr(clean_df['stock_val'])
agreement = ((clean_df['rev_val'] > 0) == (clean_df['stock_val'] > 0)).mean()
print(f"--- Dataset Statistics ---")
print(f"Revenue-Stock Correlation (r): {correlation:.3f}")
print(f"Directional Agreement: {agreement:.1%}\n")

--- Dataset Statistics ---
Revenue-Stock Correlation (r): 0.351
Directional Agreement: 68.4%



In [5]:
# 执行评估 (以 CoT 为例)
model_name = 'gemini-2.5-flash'

generation_config = {
    "temperature": 0.0,
    "response_mime_type": "application/json",
}

model = genai.GenerativeModel(
    model_name=model_name, 
    system_instruction=SYSTEM_INSTRUCTION,
    generation_config=generation_config
)

max_num = 700
mode = "qcot"      # "qcot", "cot", "normal"
threshold = 0.25

y_true, y_pred = run_experiment(twitter_df, clean_df, model=model, threshold=threshold, mode=mode, max_num=max_num)

print("\n" + "="*40)
print("FINAL EVALUATION REPORT (Stock Trend)")
print("="*40)
print(classification_report(y_true, y_pred, labels=["Growth", "Loss"]))
print(f"Accuracy Score: {accuracy_score(y_true, y_pred):.4f}")

Starting experiment in QCOT mode...
company_name: Accenture, report_period: 2011-09-01 to 2012-08-31
Company: Accenture | Year: 2012 | True: Growth | Pred: Growth | Score: 0.735
Reasoning: Accenture reported strong fiscal 2012 quarterly revenues and EPS, coupled with numerous client wins, strategic acquisitions, new service/product launches, and positive analyst recognition across all key segments.
company_name: Accenture, report_period: 2012-09-01 to 2013-08-31
Company: Accenture | Year: 2013 | True: Growth | Pred: Growth | Score: 0.335
Reasoning: Accenture demonstrates strong growth signals across its key segments, particularly in technology, public service, and financial services, driven by new solutions, client wins, and strategic investments.
company_name: Accenture, report_period: 2013-09-01 to 2014-08-31
Company: Accenture | Year: 2014 | True: Growth | Pred: Growth | Score: 0.543
Reasoning: Accenture demonstrates strong engagement and concrete project wins across all key segment